In [2]:
# ── Cell 0.1: Imports ─────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [3]:
# ── Cell 0.2: Paths & Connection ──────────────────────────────────────────────
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    PROCESSED_PATH = os.path.join("..", "data", "processed")
    TABLEAU_PATH   = os.path.join("..", "data", "tableau")
    ENV_PATH       = "../.env"
else:
    PROCESSED_PATH = os.path.join("data", "processed")
    TABLEAU_PATH   = os.path.join("data", "tableau")
    ENV_PATH       = ".env"

os.makedirs(TABLEAU_PATH, exist_ok=True)

load_dotenv(ENV_PATH)
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM bls_oews")).scalar()
print(f"✓ Connected. bls_oews: {count:,} rows")
print(f"✓ Tableau exports will go to: {os.path.abspath(TABLEAU_PATH)}")

DETAILED_FILTER = "RIGHT(occ_code, 1) != '0'"
MAJOR_FILTER    = "RIGHT(occ_code, 4) = '0000' AND occ_code != '00-0000'"

✓ Connected. bls_oews: 1,182,597 rows
✓ Tableau exports will go to: C:\Users\jabez\Documents\ai-job-displacement-observatory\data\tableau


## Export 1 — State-Level Risk Map

The BLS MSA (Metropolitan Statistical Area) names are long strings like
"New York-Newark-Jersey City, NY-NJ-PA" that Tableau can't reliably geocode
as-is. Aggregating to the **state level** is far more reliable — Tableau has
built-in geocoding for U.S. state names and abbreviations that just works.

Strategy:
1. Extract the state abbreviations from the BLS area_title string
   (e.g., "Dallas-Fort Worth, TX-OK" → ["TX", "OK"])
2. For each state abbreviation in the metro name, credit that metro's risk
   and employment to that state
3. Compute an employment-weighted average risk index per state

This gives Tableau a clean two-column table (state, risk) that it can
immediately place on a map with no manual geocoding work.

In [4]:
# ── Cell 1.1: Load Metro-Level Risk Index ────────────────────────────────────
df_risk = pd.read_csv(os.path.join(PROCESSED_PATH, "risk_index.csv"))
df_risk = df_risk.dropna(subset=['risk_index', 'tot_emp', 'area_title'])

print(f"Risk index rows: {len(df_risk):,}")
print(f"Sample area titles:")
print(df_risk['area_title'].head(5).to_string())

Risk index rows: 167,614
Sample area titles:
0    Abilene, TX
1    Abilene, TX
2    Abilene, TX
3    Abilene, TX
4    Abilene, TX


In [5]:
# ── Cell 1.2: Extract State Abbreviations from Metro Names ───────────────────
# BLS area names follow the pattern "City1-City2, ST1-ST2"
# The state abbreviations always appear after the comma.
# Example: "Chicago-Naperville-Elgin, IL-IN-WI" → ["IL", "IN", "WI"]
# We assign the metro's employment and risk to EACH state it spans,
# then take the employment-weighted average per state at the end.

def extract_states(area_title):
    """
    Returns a list of 2-letter state abbreviations from a BLS area title.
    Returns empty list if no state codes can be parsed.
    """
    try:
        # Everything after the last comma contains the state codes
        after_comma = area_title.split(',')[-1].strip()
        # State codes are separated by hyphens: "IL-IN-WI"
        codes = [s.strip() for s in after_comma.split('-')]
        # Valid state codes are exactly 2 uppercase letters
        valid = [c for c in codes if len(c) == 2 and c.isupper()]
        return valid if valid else []
    except Exception:
        return []

# Expand: one row per (metro × state) so metros spanning multiple states
# contribute to each state's average
rows = []
for _, row in df_risk.iterrows():
    states = extract_states(row['area_title'])
    for state in states:
        rows.append({
            'state_abbrev': state,
            'risk_index':   row['risk_index'],
            'tot_emp':      row['tot_emp'],
        })

df_expanded = pd.DataFrame(rows)

print(f"Expanded rows (metro × state): {len(df_expanded):,}")
print(f"Unique states found: {df_expanded['state_abbrev'].nunique()}")
print(df_expanded['state_abbrev'].value_counts().head(10).to_string())

Expanded rows (metro × state): 195,351
Unique states found: 52
state_abbrev
CA    12800
TX    11248
FL     9938
NC     7420
PA     7215
OH     6900
IN     6705
MI     6412
WI     6400
NY     6242


In [6]:
# ── Cell 1.3: Employment-Weighted Risk Index per State ────────────────────────
# weighted_risk = Σ(employment × risk) / Σ(employment)
# This correctly gives more weight to states with more workers in that metro.

df_expanded['weighted_risk'] = df_expanded['tot_emp'] * df_expanded['risk_index']

df_state = (
    df_expanded
    .groupby('state_abbrev')
    .agg(
        total_emp     = ('tot_emp',       'sum'),
        weighted_sum  = ('weighted_risk', 'sum'),
        metro_count   = ('risk_index',    'count'),
    )
    .reset_index()
)
df_state['avg_risk_index'] = df_state['weighted_sum'] / df_state['total_emp']

# Add full state names for Tableau's geocoder (it prefers names over abbreviations)
state_names = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','FL':'Florida','GA':'Georgia',
    'HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa',
    'KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland',
    'MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi',
    'MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire',
    'NJ':'New Jersey','NM':'New Mexico','NY':'New York','NC':'North Carolina',
    'ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania',
    'RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota','TN':'Tennessee',
    'TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia','WA':'Washington',
    'WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming','DC':'District of Columbia',
    'PR':'Puerto Rico',
}
df_state['state_name']  = df_state['state_abbrev'].map(state_names)
df_state['risk_tier']   = pd.cut(
    df_state['avg_risk_index'],
    bins=[0, 0.30, 0.70, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Drop rows where state name didn't map (non-standard codes)
df_state = df_state.dropna(subset=['state_name'])

print(f"States in output: {len(df_state)}")
print(f"\nTop 10 highest-risk states:")
print(df_state.nlargest(10, 'avg_risk_index')
      [['state_name', 'avg_risk_index', 'total_emp']].to_string(index=False))

States in output: 52

Top 10 highest-risk states:
    state_name  avg_risk_index  total_emp
   Puerto Rico        0.502754  1113530.0
        Nevada        0.499284  1788430.0
      Oklahoma        0.493483  1607850.0
   Mississippi        0.493424  1365020.0
         Idaho        0.492217   838150.0
       Wyoming        0.491917    97050.0
  South Dakota        0.491336   397120.0
      Arkansas        0.489852  1870830.0
     Tennessee        0.489415  3623650.0
South Carolina        0.489188  4168350.0


In [7]:
# ── Cell 1.4: Save State Risk CSV ─────────────────────────────────────────────
path = os.path.join(TABLEAU_PATH, "tableau_state_risk.csv")
df_state.to_csv(path, index=False)
print(f"✓ Saved: {path}  ({len(df_state)} rows)")
print(f"  Columns: {list(df_state.columns)}")

✓ Saved: ..\data\tableau\tableau_state_risk.csv  (52 rows)
  Columns: ['state_abbrev', 'total_emp', 'weighted_sum', 'metro_count', 'avg_risk_index', 'state_name', 'risk_tier']


## Export 2 — Occupation Risk Ranking

A clean, sorted table of occupations with all four feature scores and the
composite index. This feeds the horizontal bar chart in Worksheet 2.
I also add a human-readable major group label by looking up the 2-digit
SOC prefix — this lets me color-code bars by major occupation group in Tableau.

In [8]:
# ── Cell 2.1: Load Occupation Risk Index ─────────────────────────────────────
df_occ = pd.read_csv(os.path.join(PROCESSED_PATH, "occupation_risk_index.csv"))

# Drop rows missing the composite index
df_occ = df_occ.dropna(subset=['risk_index'])

# Add major group label from the database
major_labels = pd.read_sql(f"""
    SELECT DISTINCT
        LEFT(occ_code, 2) AS major_code,
        occ_title         AS major_group_title
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {MAJOR_FILTER}
""", engine)
major_labels['major_group_title'] = (
    major_labels['major_group_title']
    .str.replace(' Occupations', '', regex=False)
    .str.strip()
)

df_occ['major_code'] = df_occ['occ_code'].str[:2]
df_occ = df_occ.merge(major_labels, on='major_code', how='left')

# Add 2025 national employment to the occupation export
df_emp_2025 = pd.read_sql(f"""
    SELECT occ_code, tot_emp
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
""", engine)
df_emp_2025['tot_emp'] = pd.to_numeric(df_emp_2025['tot_emp'], errors='coerce')
df_emp_2025 = df_emp_2025.dropna(subset=['tot_emp'])
df_occ = df_occ.merge(df_emp_2025.rename(columns={'tot_emp': 'national_employment'}),
                       on='occ_code', how='left')

# Add risk tier label
df_occ['risk_tier'] = pd.cut(
    df_occ['risk_index'],
    bins   = [0, 0.30, 0.70, 1.0],
    labels = ['Low', 'Medium', 'High']
)

# Round all numeric columns to 4 decimal places for cleaner Tableau display
for col in ['risk_index', 'automation_prob', 'ai_exposure_score',
            'emp_trend_risk', 'wage_stagnation_risk']:
    if col in df_occ.columns:
        df_occ[col] = df_occ[col].round(4)

print(f"Occupations: {len(df_occ):,}")
print(f"\nTop 10 highest-risk:")
print(df_occ.nlargest(10, 'risk_index')
      [['occ_title', 'risk_index', 'automation_prob', 'major_group_title']]
      .to_string(index=False))

Occupations: 938

Top 10 highest-risk:
                                      occ_title  risk_index  automation_prob                 major_group_title
                            Telephone Operators      0.8727             0.97 Office and Administrative Support
              Insurance Appraisers, Auto Damage      0.8657             0.98 Business and Financial Operations
                                  Loan Officers      0.8539             0.98 Business and Financial Operations
       Credit Authorizers, Checkers, and Clerks      0.8510             0.97 Office and Administrative Support
                      Medical Transcriptionists      0.8463             0.89                Healthcare Support
                            Patternmakers, Wood      0.8430             0.91                        Production
                                Credit Analysts      0.8415             0.98 Business and Financial Operations
    Title Examiners, Abstractors, and Searchers      0.8409             0

In [9]:
export_cols = [
    'occ_code', 'occ_title', 'major_code', 'major_group_title',
    'risk_index', 'automation_prob', 'ai_exposure_score',
    'emp_trend_risk', 'wage_stagnation_risk',
    'risk_tier', 'score_source', 'education_req',
    'national_employment',    # ← added
]
df_occ_export = df_occ[[c for c in export_cols if c in df_occ.columns]]

path = os.path.join(TABLEAU_PATH, "tableau_occupation_risk.csv")
df_occ_export.to_csv(path, index=False)
print(f"✓ Saved: {path}  ({len(df_occ_export)} rows)")
print(f"  Columns: {list(df_occ_export.columns)}")

✓ Saved: ..\data\tableau\tableau_occupation_risk.csv  (938 rows)
  Columns: ['occ_code', 'occ_title', 'major_code', 'major_group_title', 'risk_index', 'automation_prob', 'ai_exposure_score', 'emp_trend_risk', 'wage_stagnation_risk', 'risk_tier', 'score_source', 'education_req', 'national_employment']


## Export 3 — Employment Trend (2018–2025)

This feeds the line chart in Worksheet 3. I export all 8 years of national
employment data for the top 50 occupations by 2025 employment size.

Why top 50 by size rather than top 50 by risk? Because the line chart is
meant to show the user how employment has evolved for whatever occupation
they select — and they're most likely to select large occupations they
recognize (Cashiers, Nurses, Software Developers) rather than obscure
high-risk ones. The dashboard's filter will let them choose any occupation.

In [10]:
# ── Cell 3.1: Load Employment Time Series (MSA → State level) ────────────────
# We pull MSA-level data this time (not national) so we have area_title
# to extract state abbreviations from. We then aggregate employment
# by state + occupation + year so the Employment Trend chart can be
# filtered by state from the map.

df_emp_msa = pd.read_sql(f"""
    SELECT year, occ_code, occ_title, area_title, tot_emp
    FROM bls_oews
    WHERE data_type = 'msa'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
    ORDER BY occ_code, year
""", engine)

df_emp_msa['tot_emp'] = pd.to_numeric(df_emp_msa['tot_emp'], errors='coerce')
df_emp_msa = df_emp_msa.dropna(subset=['tot_emp'])

# Extract primary state from area_title
# e.g. "Dallas-Fort Worth, TX-OK" → "TX"
df_emp_msa['primary_state'] = (
    df_emp_msa['area_title']
    .str.split(',').str[-1]
    .str.strip()
    .str.split('-').str[0]
    .str.strip()
)

# Aggregate to state × occupation × year
# (one metro can span multiple states — we assign to the primary state only)
df_trend = (
    df_emp_msa
    .groupby(['year', 'occ_code', 'occ_title', 'primary_state'], as_index=False)
    ['tot_emp'].sum()
)

# Select top 50 occupations by 2025 national employment
top50_codes = (
    df_trend[df_trend['year'] == 2025]
    .groupby('occ_code')['tot_emp'].sum()
    .nlargest(50)
    .index.tolist()
)
df_trend = df_trend[df_trend['occ_code'].isin(top50_codes)].copy()

# Join in risk index for color encoding
df_occ_risk = df_occ[['occ_code', 'risk_index', 'risk_tier',
                        'major_group_title']].drop_duplicates()
df_trend = df_trend.merge(df_occ_risk, on='occ_code', how='left')

# COVID flag
df_trend['is_covid_year'] = df_trend['year'] == 2020

print(f"Employment trend rows: {len(df_trend):,}")
print(f"Occupations: {df_trend['occ_code'].nunique()}")
print(f"States: {df_trend['primary_state'].nunique()}")
print(f"Years: {sorted(df_trend['year'].unique())}")
print(f"\nSample primary_state values:")
print(df_trend['primary_state'].value_counts().head(8).to_string())

Employment trend rows: 22,295
Occupations: 50
States: 52
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Sample primary_state values:
primary_state
AK    429
AL    429
AR    429
AZ    429
CA    429
CO    429
CT    429
DC    429


In [11]:
# ── Cell 3.2: Save Employment Trend CSV ──────────────────────────────────────
path = os.path.join(TABLEAU_PATH, "tableau_employment_trend.csv")
df_trend.to_csv(path, index=False)
print(f"✓ Saved: {path}  ({len(df_trend):,} rows)")
print(f"  Columns: {list(df_trend.columns)}")

✓ Saved: ..\data\tableau\tableau_employment_trend.csv  (22,295 rows)
  Columns: ['year', 'occ_code', 'occ_title', 'primary_state', 'tot_emp', 'risk_index', 'risk_tier', 'major_group_title', 'is_covid_year']


## Export 4 — Risk Tier Breakdown by Major Occupation Group

This feeds the stacked bar chart in Worksheet 4. For each of the 22 major
occupation groups, I count how many occupations fall into each risk tier
(Low / Medium / High) and sum their total 2025 national employment.

The stacked bar lets the viewer answer the question: "Which major occupation
sectors are most concentrated in the high-risk tier?" — something the
individual occupation chart can't show at a glance.

In [12]:
# ── Cell 4.1: Build Major Group × Risk Tier Table ────────────────────────────
# Load 2025 national employment to get employment totals per occupation
df_emp_2025 = pd.read_sql(f"""
    SELECT occ_code, tot_emp
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
""", engine)
df_emp_2025['tot_emp'] = pd.to_numeric(df_emp_2025['tot_emp'], errors='coerce')
df_emp_2025 = df_emp_2025.dropna(subset=['tot_emp'])

# Join employment to the occupation risk index
df_breakdown = df_occ.merge(df_emp_2025, on='occ_code', how='inner')
df_breakdown = df_breakdown.dropna(subset=['risk_tier', 'major_group_title'])

# Aggregate: count occupations and sum employment per (major group × risk tier)
df_grouped = (
    df_breakdown
    .groupby(['major_group_title', 'risk_tier'], observed=True)
    .agg(
        occupation_count = ('occ_code', 'nunique'),
        total_employment = ('tot_emp',  'sum'),
    )
    .reset_index()
)

# Add total employment per major group for sorting
group_totals = (
    df_grouped
    .groupby('major_group_title')['total_employment']
    .sum()
    .reset_index()
    .rename(columns={'total_employment': 'group_total_emp'})
)
df_grouped = df_grouped.merge(group_totals, on='major_group_title', how='left')

# Compute percentage of each group's employment in each risk tier
df_grouped['pct_of_group'] = (
    df_grouped['total_employment'] / df_grouped['group_total_emp'] * 100
).round(2)

print(f"Output rows: {len(df_grouped)}")
print(f"\nSample (sorted by high-risk employment):")
print(df_grouped[df_grouped['risk_tier'] == 'High']
      .nlargest(8, 'total_employment')
      [['major_group_title', 'occupation_count', 'total_employment', 'pct_of_group']]
      .to_string(index=False))

Output rows: 48

Sample (sorted by high-risk employment):
                  major_group_title  occupation_count  total_employment  pct_of_group
  Office and Administrative Support                35           9961500         55.80
  Business and Financial Operations                 9           2409680         13.94
                         Production                40           1618970         22.05
                  Sales and Related                 5           1615550         11.78
 Transportation and Material Moving                11            197670          1.19
       Architecture and Engineering                 3            171060          2.64
        Construction and Extraction                 5             78570          1.00
Educational Instruction and Library                 1             68690          0.75


In [13]:
# ── Cell 4.2: Save Risk Tier Breakdown CSV ───────────────────────────────────
path = os.path.join(TABLEAU_PATH, "tableau_major_group_risk_tiers.csv")
df_grouped.to_csv(path, index=False)
print(f"✓ Saved: {path}  ({len(df_grouped)} rows)")
print(f"  Columns: {list(df_grouped.columns)}")

✓ Saved: ..\data\tableau\tableau_major_group_risk_tiers.csv  (48 rows)
  Columns: ['major_group_title', 'risk_tier', 'occupation_count', 'total_employment', 'group_total_emp', 'pct_of_group']


In [14]:
# ── Cell 5.1: Verify All 4 Tableau Exports ───────────────────────────────────
print("=== TABLEAU DATA PREP — EXPORT VERIFICATION ===\n")

exports = {
    "tableau_state_risk.csv":             "Worksheet 1 — State choropleth map",
    "tableau_occupation_risk.csv":        "Worksheet 2 — Occupation risk ranking",
    "tableau_employment_trend.csv":       "Worksheet 3 — Employment trend lines",
    "tableau_major_group_risk_tiers.csv": "Worksheet 4 — Risk tier stacked bars",
}

all_good = True
for filename, description in exports.items():
    path = os.path.join(TABLEAU_PATH, filename)
    if os.path.exists(path):
        df_check = pd.read_csv(path)
        print(f"  ✓  {filename}")
        print(f"       {description}")
        print(f"       {len(df_check):,} rows × {len(df_check.columns)} columns")
    else:
        print(f"  ✗  MISSING: {filename}")
        all_good = False
    print()

print(f"{'✓ All exports ready for Tableau.' if all_good else '✗ Fix missing files above.'}")
print(f"\nExport folder: {os.path.abspath(TABLEAU_PATH)}")

=== TABLEAU DATA PREP — EXPORT VERIFICATION ===

  ✓  tableau_state_risk.csv
       Worksheet 1 — State choropleth map
       52 rows × 7 columns

  ✓  tableau_occupation_risk.csv
       Worksheet 2 — Occupation risk ranking
       938 rows × 13 columns

  ✓  tableau_employment_trend.csv
       Worksheet 3 — Employment trend lines
       22,295 rows × 9 columns

  ✓  tableau_major_group_risk_tiers.csv
       Worksheet 4 — Risk tier stacked bars
       48 rows × 6 columns

✓ All exports ready for Tableau.

Export folder: C:\Users\jabez\Documents\ai-job-displacement-observatory\data\tableau


## Prep Notebook Summary

This notebook exports 4 CSV files to `data/tableau/` for use in Tableau Public:

| File | Rows | Purpose |
|---|---|---|
| `tableau_state_risk.csv` | ~48 | Employment-weighted avg risk index per U.S. state |
| `tableau_occupation_risk.csv` | ~800 | All occupations with 4-component risk index |
| `tableau_employment_trend.csv` | ~400 | 2018–2025 employment for top 50 occupations |
| `tableau_major_group_risk_tiers.csv` | ~66 | Occupation count + employment by major group × risk tier |

### Why pre-aggregate in Python instead of letting Tableau do it?
Tableau can aggregate data internally, but pre-aggregating in Python:
- Makes the workbook open faster (less computation at render time)
- Gives you full control over exactly how metrics are calculated
- Keeps the Tableau workbook simpler and easier to maintain
- Ensures the numbers in Tableau exactly match the numbers in your notebooks

In [15]:
import pandas as pd, os

path = os.path.join(TABLEAU_PATH, "tableau_occupation_risk.csv")
df_check = pd.read_csv(path)
print("Columns in CSV:")
print(list(df_check.columns))
print(f"\nRows: {len(df_check)}")

Columns in CSV:
['occ_code', 'occ_title', 'major_code', 'major_group_title', 'risk_index', 'automation_prob', 'ai_exposure_score', 'emp_trend_risk', 'wage_stagnation_risk', 'risk_tier', 'score_source', 'education_req', 'national_employment']

Rows: 938


In [16]:
# Quick verify — run this after Cell 2.2
df_verify = pd.read_csv(os.path.join(TABLEAU_PATH, "tableau_occupation_risk.csv"))
print("Columns:", list(df_verify.columns))
print(f"national_employment nulls: {df_verify['national_employment'].isna().sum()}")
print(f"\nSample:")
print(df_verify[['occ_title', 'risk_index', 'national_employment']].head(5).to_string(index=False))

Columns: ['occ_code', 'occ_title', 'major_code', 'major_group_title', 'risk_index', 'automation_prob', 'ai_exposure_score', 'emp_trend_risk', 'wage_stagnation_risk', 'risk_tier', 'score_source', 'education_req', 'national_employment']
national_employment nulls: 0

Sample:
                          occ_title  risk_index  national_employment
                   Chief Executives      0.3197               204350
                   Chief Executives      0.3026               204350
    General and Operations Managers      0.4043              3503020
Advertising and Promotions Managers      0.4432                21470
                 Marketing Managers      0.2741               395240


In [17]:
# ── KPI Strip Values ──────────────────────────────────────────────────────────
import pandas as pd
import os

# Load the processed outputs
df_metro = pd.read_csv(os.path.join(PROCESSED_PATH, "risk_index.csv"))
df_occ   = pd.read_csv(os.path.join(PROCESSED_PATH, "occupation_risk_index.csv"))

# KPI 1: Total workers covered
total_workers = df_metro['tot_emp'].sum()
print(f"KPI 1 — Total Workers Covered: {total_workers/1e6:.1f}M")

# KPI 2: % in high-risk occupations (employment-weighted)
df_metro['auto_x_emp'] = df_metro['tot_emp'] * df_metro['automation_prob']
weighted_risk = df_metro['auto_x_emp'].sum() / df_metro['tot_emp'].sum()
high_risk_emp = df_metro[df_metro['automation_prob'] > 0.70]['tot_emp'].sum()
high_risk_pct = high_risk_emp / df_metro['tot_emp'].sum() * 100
print(f"KPI 2 — Workers in High-Risk Occupations: {high_risk_pct:.0f}%")

# KPI 3: Occupations scored
total_occs  = len(df_occ)
actual_occs = (df_occ['score_source'] == 'actual').sum() if 'score_source' in df_occ.columns else total_occs
print(f"KPI 3 — Total Occupations Scored: {total_occs:,}")
print(f"         (of which {actual_occs} have actual Frey-Osborne scores)")

# KPI 4: Metro areas analyzed
total_metros = df_metro['area_code'].nunique()
print(f"KPI 4 — Metro Areas Analyzed: {total_metros:,}")

# Bonus stats you might want to use
print(f"\n── Bonus Stats ──")
print(f"Years of BLS data: 2018–2025 (8 years)")
print(f"LLM briefings generated: check llm_briefings table")
print(f"Avg risk index (employment-weighted): {weighted_risk:.3f}")

KPI 1 — Total Workers Covered: 167.0M
KPI 2 — Workers in High-Risk Occupations: 37%
KPI 3 — Total Occupations Scored: 938
         (of which 693 have actual Frey-Osborne scores)
KPI 4 — Metro Areas Analyzed: 393

── Bonus Stats ──
Years of BLS data: 2018–2025 (8 years)
LLM briefings generated: check llm_briefings table
Avg risk index (employment-weighted): 0.450


In [18]:
# LLM briefings count from database
with engine.connect() as conn:
    briefing_count = conn.execute(text("SELECT COUNT(*) FROM llm_briefings")).scalar()
print(f"KPI 5 — LLM Risk Briefings Generated: {briefing_count}")

KPI 5 — LLM Risk Briefings Generated: 0


In [19]:
import pandas as pd, os
df_check = pd.read_csv(os.path.join(TABLEAU_PATH, "tableau_employment_trend.csv"))
print(list(df_check.columns))
print(f"Rows: {len(df_check):,}")
print(f"primary_state sample: {df_check['primary_state'].dropna().head(5).tolist()}")

['year', 'occ_code', 'occ_title', 'primary_state', 'tot_emp', 'risk_index', 'risk_tier', 'major_group_title', 'is_covid_year']
Rows: 22,295
primary_state sample: ['AK', 'AL', 'AR', 'AZ', 'CA']


In [20]:
# ── UNIFIED DATA SOURCE — Single CSV for all Tableau worksheets ───────────────
# All four worksheets will be built from this one file.
# Granularity: occupation × year × state (most detailed level)
# Tableau aggregates up to whatever level each worksheet needs.

# Start with MSA employment data — this gives us year, occupation, and state
df_unified_base = pd.read_sql(f"""
    SELECT year, occ_code, occ_title, area_title, area_code, tot_emp
    FROM bls_oews
    WHERE data_type = 'msa'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
      AND year IN (2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025)
    ORDER BY occ_code, year
""", engine)

df_unified_base['tot_emp'] = pd.to_numeric(df_unified_base['tot_emp'], errors='coerce')
df_unified_base = df_unified_base.dropna(subset=['tot_emp'])

# Extract primary state from area_title
df_unified_base['primary_state'] = (
    df_unified_base['area_title']
    .str.split(',').str[-1]
    .str.strip()
    .str.split('-').str[0]
    .str.strip()
)

print(f"Base rows: {len(df_unified_base):,}")
print(f"Occupations: {df_unified_base['occ_code'].nunique()}")
print(f"States: {df_unified_base['primary_state'].nunique()}")

Base rows: 1,034,011
Occupations: 925
States: 52


In [21]:
# ── Join all feature data onto the base ───────────────────────────────────────

# 1. Occupation risk index (automation_prob, risk_index, risk_tier, etc.)
df_occ_features = pd.read_csv(os.path.join(PROCESSED_PATH, "occupation_risk_index.csv"))
df_occ_features['major_code'] = df_occ_features['occ_code'].str[:2]

# Get major group labels
major_labels = pd.read_sql(f"""
    SELECT DISTINCT LEFT(occ_code, 2) AS major_code,
           occ_title AS major_group_title
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {MAJOR_FILTER}
""", engine)
major_labels['major_group_title'] = (
    major_labels['major_group_title']
    .str.replace(' Occupations', '', regex=False).str.strip()
)
df_occ_features = df_occ_features.merge(major_labels, on='major_code', how='left')

# Compute risk tier
df_occ_features['risk_tier'] = pd.cut(
    df_occ_features['risk_index'],
    bins=[0, 0.30, 0.70, 1.0],
    labels=['Low', 'Medium', 'High']
).astype(str)

occ_cols = ['occ_code', 'risk_index', 'automation_prob', 'ai_exposure_score',
            'emp_trend_risk', 'wage_stagnation_risk', 'risk_tier',
            'major_code', 'major_group_title', 'education_req', 'score_source']

df_unified = df_unified_base.merge(
    df_occ_features[occ_cols],
    on='occ_code', how='left'
)

# 2. State-level risk (avg_risk_index per state)
state_names = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','FL':'Florida','GA':'Georgia',
    'HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa',
    'KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland',
    'MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi',
    'MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire',
    'NJ':'New Jersey','NM':'New Mexico','NY':'New York','NC':'North Carolina',
    'ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania',
    'RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota','TN':'Tennessee',
    'TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia','WA':'Washington',
    'WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming','DC':'District of Columbia',
}
df_unified['state_name'] = df_unified['primary_state'].map(state_names)

# 3. Add is_covid_year flag
df_unified['is_covid_year'] = df_unified['year'] == 2020

# 4. Add 2025 national employment for bar width encoding
df_nat_emp = pd.read_sql(f"""
    SELECT occ_code, tot_emp AS national_employment
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
""", engine)
df_nat_emp['national_employment'] = pd.to_numeric(
    df_nat_emp['national_employment'], errors='coerce'
)
df_unified = df_unified.merge(df_nat_emp, on='occ_code', how='left')

print(f"\nUnified dataset: {len(df_unified):,} rows × {len(df_unified.columns)} columns")
print(f"Columns: {list(df_unified.columns)}")
print(f"\nNull counts in key fields:")
for col in ['risk_index', 'risk_tier', 'major_group_title', 'state_name']:
    print(f"  {col}: {df_unified[col].isna().sum()} nulls")


Unified dataset: 1,258,540 rows × 20 columns
Columns: ['year', 'occ_code', 'occ_title', 'area_title', 'area_code', 'tot_emp', 'primary_state', 'risk_index', 'automation_prob', 'ai_exposure_score', 'emp_trend_risk', 'wage_stagnation_risk', 'risk_tier', 'major_code', 'major_group_title', 'education_req', 'score_source', 'state_name', 'is_covid_year', 'national_employment']

Null counts in key fields:
  risk_index: 26328 nulls
  risk_tier: 26328 nulls
  major_group_title: 26328 nulls
  state_name: 155724 nulls


In [22]:
# ── Save unified CSV ───────────────────────────────────────────────────────────
unified_path = os.path.join(TABLEAU_PATH, "tableau_unified.csv")
df_unified.to_csv(unified_path, index=False)

print(f"✓ Saved: {unified_path}")
print(f"  Rows: {len(df_unified):,}")
print(f"  Size: {os.path.getsize(unified_path) / 1e6:.1f} MB")
print(f"\nThis single file replaces all four separate Tableau CSVs.")
print(f"All four worksheets will be rebuilt from this one source.")

✓ Saved: ..\data\tableau\tableau_unified.csv
  Rows: 1,258,540
  Size: 311.4 MB

This single file replaces all four separate Tableau CSVs.
All four worksheets will be rebuilt from this one source.


In [23]:
# ── Add Prophet Forecasts to Employment Trend Data ────────────────────────────
# We already ran Prophet in Part 6 and saved forecasts to employment_forecasts.csv
# Now we pipe those predictions into the Tableau data as additional rows,
# flagged as is_forecast=True so Tableau can style them differently.
# This directly connects Part 6 ML work to the dashboard.

import pandas as pd
import os

# Load the Prophet forecasts generated in Part 6
forecast_path = os.path.join(PROCESSED_PATH, "employment_forecasts.csv")
df_forecasts  = pd.read_csv(forecast_path)

print(f"Forecast rows loaded: {len(df_forecasts):,}")
print(f"Columns: {list(df_forecasts.columns)}")
print(f"Years in forecast: {sorted(df_forecasts['year'].unique())}")
print(f"Occupations: {df_forecasts['occ_code'].nunique()}")

Forecast rows loaded: 193
Columns: ['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper', 'additive_terms', 'additive_terms_lower', 'additive_terms_upper', 'covid_anomaly', 'covid_anomaly_lower', 'covid_anomaly_upper', 'holidays', 'holidays_lower', 'holidays_upper', 'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper', 'yhat', 'occ_code', 'occ_title', 'year', 'actual']
Years in forecast: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027)]
Occupations: 20


In [24]:
# ── Format Forecast Rows to Match tableau_unified ─────────────────────────────
# Prophet output has: occ_code, occ_title, ds, yhat, yhat_lower, yhat_upper,
# year, actual
# We need to create rows that match the structure of tableau_unified so
# Tableau can blend them seamlessly with historical data.

# Filter to ONLY the true forecast years (2026-2027)
# 2018-2025 rows already exist in tableau_unified as actual data
df_future = df_forecasts[df_forecasts['year'].isin([2026, 2027])].copy()

# Get occupation risk info to join in
df_occ_info = pd.read_csv(os.path.join(PROCESSED_PATH, "occupation_risk_index.csv"))
df_occ_info['major_code'] = df_occ_info['occ_code'].str[:2]

# Get major group labels
major_labels = pd.read_sql(f"""
    SELECT DISTINCT LEFT(occ_code, 2) AS major_code,
           occ_title AS major_group_title
    FROM bls_oews
    WHERE year = 2025 AND data_type = 'national'
      AND {MAJOR_FILTER}
""", engine)
major_labels['major_group_title'] = (
    major_labels['major_group_title']
    .str.replace(' Occupations', '', regex=False).str.strip()
)
df_occ_info = df_occ_info.merge(major_labels, on='major_code', how='left')

# Add risk tier
df_occ_info['risk_tier'] = pd.cut(
    df_occ_info['risk_index'],
    bins=[0, 0.30, 0.70, 1.0],
    labels=['Low', 'Medium', 'High']
).astype(str)

# Build the forecast rows with the same columns as tableau_unified
df_forecast_rows = df_future.merge(
    df_occ_info[['occ_code', 'risk_index', 'risk_tier', 'major_group_title',
                  'automation_prob', 'ai_exposure_score', 'emp_trend_risk',
                  'wage_stagnation_risk', 'education_req', 'score_source']],
    on='occ_code', how='left'
)

# Rename Prophet columns to match unified schema
df_forecast_rows = df_forecast_rows.rename(columns={
    'yhat':       'tot_emp',        # predicted employment = tot_emp for Tableau
    'yhat_lower': 'forecast_lower', # confidence interval lower bound
    'yhat_upper': 'forecast_upper', # confidence interval upper bound
})

# Add flag columns
df_forecast_rows['is_forecast']   = True
df_forecast_rows['is_covid_year'] = False

# Add placeholder geography columns that exist in tableau_unified
df_forecast_rows['area_code']      = 'NATIONAL'
df_forecast_rows['area_title']     = 'National'
df_forecast_rows['primary_state']  = 'US'
df_forecast_rows['state_name']     = 'United States'
df_forecast_rows['national_employment'] = df_forecast_rows['tot_emp']

print(f"Forecast rows to append: {len(df_forecast_rows)}")
print(f"Years: {sorted(df_forecast_rows['year'].unique())}")
print(f"Occupations: {df_forecast_rows['occ_code'].nunique()}")
print(f"\nSample forecast values:")
print(df_forecast_rows[['occ_title', 'year', 'tot_emp', 'forecast_lower',
                          'forecast_upper', 'is_forecast']].head(6).to_string(index=False))

Forecast rows to append: 52
Years: [np.int64(2026), np.int64(2027)]
Occupations: 20

Sample forecast values:
                    occ_title  year      tot_emp  forecast_lower  forecast_upper  is_forecast
          Retail Salespersons  2026 3.643499e+06    3.048306e+06    4.161953e+06         True
          Retail Salespersons  2027 3.635928e+06    2.631279e+06    4.419055e+06         True
Fast Food and Counter Workers  2026 3.068512e+06    2.710217e+06    3.416779e+06         True
Fast Food and Counter Workers  2026 3.068512e+06    2.710217e+06    3.416779e+06         True
Fast Food and Counter Workers  2027 2.972856e+06    2.603340e+06    3.314244e+06         True
Fast Food and Counter Workers  2027 2.972856e+06    2.603340e+06    3.314244e+06         True


In [25]:
# ── Rebuild tableau_unified with Forecast Rows Appended ───────────────────────
# Load current unified dataset
unified_path = os.path.join(TABLEAU_PATH, "tableau_unified.csv")
df_unified   = pd.read_csv(unified_path)

# Add missing columns to historical data so schemas match
df_unified['is_forecast']    = False
df_unified['forecast_lower'] = None   # no confidence interval for actual data
df_unified['forecast_upper'] = None

print(f"Historical rows: {len(df_unified):,}")

# Align columns between historical and forecast dataframes
# Only keep columns that exist in both
shared_cols = [c for c in df_unified.columns if c in df_forecast_rows.columns]
df_forecast_aligned = df_forecast_rows[shared_cols].copy()

# Append forecast rows
df_unified_with_forecasts = pd.concat(
    [df_unified, df_forecast_aligned],
    ignore_index=True
)

print(f"After appending forecasts: {len(df_unified_with_forecasts):,} rows")
print(f"Years now in dataset: {sorted(df_unified_with_forecasts['year'].unique())}")
print(f"is_forecast value counts:")
print(df_unified_with_forecasts['is_forecast'].value_counts())

# Save
df_unified_with_forecasts.to_csv(unified_path, index=False)
print(f"\n✓ Saved updated tableau_unified.csv")

Historical rows: 1,258,540
After appending forecasts: 1,258,592 rows
Years now in dataset: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027)]
is_forecast value counts:
is_forecast
False    1258540
True          52
Name: count, dtype: int64

✓ Saved updated tableau_unified.csv


In [4]:
import pandas as pd, os

unified_path = os.path.join(TABLEAU_PATH, "tableau_unified.csv")
df_check = pd.read_csv(unified_path)

print("Years currently in tableau_unified:")
print(sorted(df_check['year'].unique()))

Years currently in tableau_unified:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027)]


In [6]:
# ── Clean Rebuild: Remove All Forecast Rows, Start Fresh ──────────────────────
import pandas as pd, os

unified_path = os.path.join(TABLEAU_PATH, "tableau_unified.csv")
df_unified   = pd.read_csv(unified_path)

# Remove any existing 2026/2027 rows (the duplicated forecast attempts)
df_unified_clean = df_unified[~df_unified['year'].isin([2026, 2027])].copy()

print(f"Rows before cleanup: {len(df_unified):,}")
print(f"Rows after removing 2026/2027: {len(df_unified_clean):,}")
print(f"Years remaining: {sorted(df_unified_clean['year'].unique())}")

# Overwrite with the clean version
df_unified_clean.to_csv(unified_path, index=False)
print("\n✓ Saved clean base file")

Rows before cleanup: 1,258,592
Rows after removing 2026/2027: 1,258,540
Years remaining: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

✓ Saved clean base file


In [7]:
# ── Check Granularity ──────────────────────────────────────────────────────────
sample = df_unified_clean[
    (df_unified_clean['occ_code'] == '11-1011') &
    (df_unified_clean['year'] == 2018)
]
print(f"Rows for occ_code=11-1011, year=2018: {len(sample)}")
print(f"Sum of tot_emp across those rows: {sample['tot_emp'].sum():,.0f}")
print(f"\nColumns that vary across these rows:")
for col in ['area_code', 'area_title', 'primary_state', 'state_name']:
    if col in sample.columns:
        print(f"  {col}: {sample[col].nunique()} unique values")

Rows for occ_code=11-1011, year=2018: 654
Sum of tot_emp across those rows: 303,680

Columns that vary across these rows:
  area_code: 327 unique values
  area_title: 0 unique values
  primary_state: 0 unique values
  state_name: 0 unique values


In [8]:
# ── Add Forecast Rows at Correct Granularity ──────────────────────────────────
forecast_path = os.path.join(PROCESSED_PATH, "employment_forecasts.csv")
df_forecast   = pd.read_csv(forecast_path)

df_future = df_forecast[df_forecast['year'].isin([2026, 2027])][
    ['occ_code', 'occ_title', 'year', 'yhat']
].copy()
df_future = df_future.rename(columns={'yhat': 'tot_emp'})

# One row per occupation per forecast year = the full national total
# (matches how SUM(Tot Emp) would aggregate historical metro rows)
df_reference = (
    df_unified_clean[df_unified_clean['year'] == 2025]
    .drop_duplicates(subset='occ_code')
)
static_cols = [c for c in df_unified_clean.columns if c not in
               ['year', 'tot_emp', 'occ_title', 'occ_code',
                'area_code', 'area_title', 'primary_state', 'state_name']]

df_future_full = df_future.merge(
    df_reference[['occ_code'] + static_cols],
    on='occ_code', how='left'
)

# Flag these clearly as forecast-only, single national rows
df_future_full['area_code']     = 'FORECAST'
df_future_full['area_title']    = 'National Forecast'
df_future_full['primary_state'] = 'US'
df_future_full['state_name']    = 'United States'

df_future_full = df_future_full[df_unified_clean.columns.tolist()]

df_final = pd.concat([df_unified_clean, df_future_full], ignore_index=True)
df_final.to_csv(unified_path, index=False)

print(f"✓ Final row count: {len(df_final):,}")
print(f"Years: {sorted(df_final['year'].unique())}")

# Verify: total employment per year for one occupation, should be smooth
check = df_final[df_final['occ_code'] == '11-1011'].groupby('year')['tot_emp'].sum()
print(f"\nNational total by year for occ_code 11-1011:")
print(check.to_string())

✓ Final row count: 1,258,580
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027)]

National total by year for occ_code 11-1011:
year
2018    303680.0
2019    277800.0
2020    277980.0
2021    308940.0
2022    292480.0
2023    278700.0
2024    294720.0
2025    270300.0


In [9]:
# ── Verify a Chart-Relevant Occupation ─────────────────────────────────────────
df_final = pd.read_csv(unified_path)

for code, name in [('41-2011', 'Cashiers'), ('29-1141', 'Registered Nurses'),
                    ('15-1252', 'Software Developers')]:
    check = df_final[df_final['occ_code'] == code].groupby('year')['tot_emp'].sum()
    print(f"\n{name} ({code}):")
    print(check.to_string())
    


Cashiers (41-2011):
year
2018    3.124580e+06
2019    3.096690e+06
2020    2.847600e+06
2021    2.829580e+06
2022    2.802020e+06
2023    2.799570e+06
2024    2.668090e+06
2025    2.615670e+06
2026    3.264634e+06
2027    3.254720e+06

Registered Nurses (29-1141):
year
2018    1.314590e+07
2019    1.323780e+07
2020    1.328235e+07
2021    1.366435e+07
2022    1.365140e+07
2023    1.407470e+07
2024    1.461430e+07
2025    1.492555e+07
2026    3.346636e+06
2027    3.408480e+06

Software Developers (15-1252):
year
2021    1.230760e+06
2022    1.350570e+06
2023    1.442680e+06
2024    1.413840e+06
2025    1.441640e+06
2026    2.023485e+06
2027    2.145575e+06


In [10]:
# ── Diagnose the Forecast Source Data Directly ─────────────────────────────────
df_forecast_raw = pd.read_csv(os.path.join(PROCESSED_PATH, "employment_forecasts.csv"))

for code, name in [('41-2011', 'Cashiers'), ('29-1141', 'Registered Nurses'),
                    ('15-1252', 'Software Developers')]:
    rows = df_forecast_raw[df_forecast_raw['occ_code'] == code]
    print(f"\n{name} ({code}) — rows in employment_forecasts.csv: {len(rows)}")
    if len(rows) > 0:
        print(rows[['year', 'yhat']].to_string(index=False))
    else:
        print("  NOT FOUND with exact match — checking for whitespace/case issues:")
        close = df_forecast_raw[df_forecast_raw['occ_code'].str.strip() == code.strip()]
        print(f"  Found {len(close)} rows after stripping whitespace")


Cashiers (41-2011) — rows in employment_forecasts.csv: 10
 year         yhat
 2018 3.635774e+06
 2019 3.596175e+06
 2020 3.333100e+06
 2021 3.314231e+06
 2022 3.304317e+06
 2023 3.294403e+06
 2024 3.284462e+06
 2025 3.274548e+06
 2026 3.264634e+06
 2027 3.254720e+06

Registered Nurses (29-1141) — rows in employment_forecasts.csv: 10
 year         yhat
 2018 2.952258e+06
 2019 2.980562e+06
 2020 2.986567e+06
 2021 3.037248e+06
 2022 3.099092e+06
 2023 3.160935e+06
 2024 3.222949e+06
 2025 3.284793e+06
 2026 3.346636e+06
 2027 3.408480e+06

Software Developers (15-1252) — rows in employment_forecasts.csv: 7
 year         yhat
 2021 1.364180e+06
 2022 1.534790e+06
 2023 1.656880e+06
 2024 1.779305e+06
 2025 1.901395e+06
 2026 2.023485e+06
 2027 2.145575e+06


In [11]:
# ── Fix Duplication in tableau_unified ─────────────────────────────────────────
import pandas as pd, os

unified_path = os.path.join(TABLEAU_PATH, "tableau_unified.csv")
df_raw = pd.read_csv(unified_path)

print(f"Rows before dedup: {len(df_raw):,}")

# Drop exact duplicate rows entirely — every column identical means it's a
# true duplicate, not two different metros
df_deduped = df_raw.drop_duplicates()

print(f"Rows after exact dedup: {len(df_deduped):,}")

# Verify the fix on Registered Nurses 2025
check = (
    df_deduped[(df_deduped['occ_code'] == '29-1141') & (df_deduped['year'] == 2025)]
    ['tot_emp'].sum()
)
print(f"\nRegistered Nurses 2025 total after dedup: {check:,.0f}")
print(f"True BLS value should be close to: 3,379,720")

Rows before dedup: 1,258,580
Rows after exact dedup: 1,258,580

Registered Nurses 2025 total after dedup: 14,925,550
True BLS value should be close to: 3,379,720


In [12]:
# ── Build a Clean, Standalone Employment Trend Export ──────────────────────────
import pandas as pd, os

# Pull TRUE national employment directly from source — no joins, no merges
df_trend_clean = pd.read_sql(f"""
    SELECT year, occ_code, occ_title, tot_emp
    FROM bls_oews
    WHERE data_type = 'national'
      AND {DETAILED_FILTER}
      AND tot_emp IS NOT NULL
    ORDER BY occ_code, year
""", engine)
df_trend_clean['tot_emp'] = pd.to_numeric(df_trend_clean['tot_emp'], errors='coerce')
df_trend_clean = df_trend_clean.dropna(subset=['tot_emp'])

# Restrict to the occupations Prophet actually forecasted (already verified correct)
df_forecast_raw = pd.read_csv(os.path.join(PROCESSED_PATH, "employment_forecasts.csv"))
forecasted_codes = df_forecast_raw['occ_code'].unique().tolist()

df_trend_clean = df_trend_clean[df_trend_clean['occ_code'].isin(forecasted_codes)]

# Append the verified-correct 2026/2027 Prophet predictions
df_future = df_forecast_raw[df_forecast_raw['year'].isin([2026, 2027])][
    ['occ_code', 'occ_title', 'year', 'yhat']
].rename(columns={'yhat': 'tot_emp'})

df_trend_final = pd.concat([df_trend_clean, df_future], ignore_index=True)
df_trend_final['is_forecast'] = df_trend_final['year'] >= 2026

# COVID flag for reference line context
df_trend_final['is_covid_year'] = df_trend_final['year'] == 2020

# Save as its own dedicated file — separate from tableau_unified
clean_path = os.path.join(TABLEAU_PATH, "tableau_employment_trend_clean.csv")
df_trend_final.to_csv(clean_path, index=False)

print(f"✓ Saved: {clean_path}")
print(f"Rows: {len(df_trend_final)}")
print(f"Occupations: {df_trend_final['occ_code'].nunique()}")

# Verify Registered Nurses one more time
check = df_trend_final[df_trend_final['occ_code'] == '29-1141'][['year', 'tot_emp']].sort_values('year')
print(f"\nRegistered Nurses — full clean timeline:")
print(check.to_string(index=False))

✓ Saved: ..\data\tableau\tableau_employment_trend_clean.csv
Rows: 193
Occupations: 20

Registered Nurses — full clean timeline:
 year      tot_emp
 2018 2.951960e+06
 2019 2.982280e+06
 2020 2.986500e+06
 2021 3.047530e+06
 2022 3.072700e+06
 2023 3.175390e+06
 2024 3.282010e+06
 2025 3.379720e+06
 2026 3.346636e+06
 2027 3.408480e+06


In [13]:
# ── Check for Occ Title Mismatches Around the Gap ──────────────────────────────
df_trend_final = pd.read_csv(os.path.join(TABLEAU_PATH, "tableau_employment_trend_clean.csv"))

for code in ['41-2011', '29-1141']:  # Cashiers, Registered Nurses — adjust as needed
    sub = df_trend_final[df_trend_final['occ_code'] == code]
    print(f"\n{code} — distinct Occ Title values used across all years:")
    print(sub.groupby('year')['occ_title'].first().to_string())


41-2011 — distinct Occ Title values used across all years:
year
2018    Cashiers
2019    Cashiers
2020    Cashiers
2021    Cashiers
2022    Cashiers
2023    Cashiers
2024    Cashiers
2025    Cashiers
2026    Cashiers
2027    Cashiers

29-1141 — distinct Occ Title values used across all years:
year
2018    Registered Nurses
2019    Registered Nurses
2020    Registered Nurses
2021    Registered Nurses
2022    Registered Nurses
2023    Registered Nurses
2024    Registered Nurses
2025    Registered Nurses
2026    Registered Nurses
2027    Registered Nurses
